In [10]:
from pathlib import Path
import sys
import pandas as pd
import yaml

root_path = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "utils").exists()), None)
if root_path is None:
    raise FileNotFoundError("Could not locate project root containing 'utils' directory")
sys.path.append(str(root_path))

from utils.helpers import save_csv_file, load_data
from utils.helper_CBM import (
    get_cbm_valid_questions,
    cast_columns_to_string,
    filter_cbm_questions,
    drop_columns_from_config,
    add_auto_score,
    split_response_cbm,
    map_confidence_level,
    validate_and_remove_invalid_confidence
)


In [11]:
df = load_data("../data/cleaned_data/IN1020_2025_exam_cleaned_data.csv")
df.head()

Data loaded successfully from ../data/cleaned_data/IN1020_2025_exam_cleaned_data.csv


,exam_start_time,exam_end_time,extra_time_mins,incident_time_mins,student_id,max_question_score,question_number,question_title,question_duration_sec,auto_score_per_question,candidate_response_code,candidate_response_text,response_CBM,total_score
0,2025-12-08 14:08:05,2025-12-08 15:17:51,0,0,363,2.0,1.1,IN1020 H25 bit-representasjon,126,1.0,simpleChoice_1368096164453,171,1,53.82
1,2025-12-08 14:08:05,2025-12-08 15:17:51,0,0,363,2.5,1.2,IN1020 H25 Tallsystemer og 10,107,0.5,simpleChoice_1368096164453,simpleChoice_1368096164453,1,53.82
2,2025-12-08 14:08:05,2025-12-08 15:17:51,0,0,363,3.5,1.3,IN1020 H25 Tallsystemer representasjon,26,1.5,simpleChoice_1368096164459,Heksadesimale tall,3,53.82
3,2025-12-08 14:08:05,2025-12-08 15:17:51,0,0,363,3.5,1.3,IN1020 H25 Tallsystemer representasjon,26,1.5,simpleChoice_IA17239952044953d28e157-ac4c-49db...,Base 64 (64-tallsystemet),8,53.82
4,2025-12-08 14:08:05,2025-12-08 15:17:51,0,0,363,3.5,1.3,IN1020 H25 Tallsystemer representasjon,26,1.5,simpleChoice_IA1723995263688fff30f23-beb3-435e...,Binære tall,9,53.82


In [12]:
config_path = root_path / "configs" / "CBM_config.yaml"
with open(config_path, 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)
print(f"Config loaded from {config_path}")


Config loaded from /Users/yulis/Desktop/Masteroppgave_Eksamensanalyse/configs/CBM_config.yaml


In [13]:
df = cast_columns_to_string(df, config.get('string_columns', []))

valid_questions = get_cbm_valid_questions(prefix="2.", start=1, end=7)
df_CBM = filter_cbm_questions(df, valid_questions)
df_CBM = drop_columns_from_config(df_CBM, config.get('columns_to_remove', []))

df_CBM.info()

<class 'pandas.core.frame.DataFrame'>
Index: 17015 entries, 11 to 68856
Data columns (total 8 columns):
 #   Column                   Non-Null Count  Dtype   
---  ------                   --------------  -----   
 0   student_id               17015 non-null  int64   
 1   max_question_score       17015 non-null  float64 
 2   question_number          17015 non-null  category
 3   question_title           17015 non-null  object  
 4   question_duration_sec    17015 non-null  int64   
 5   auto_score_per_question  17015 non-null  float64 
 6   response_CBM             17015 non-null  object  
 7   total_score              17015 non-null  float64 
dtypes: category(1), float64(3), int64(2), object(2)
memory usage: 1.1+ MB


In [14]:
df_CBM = add_auto_score(df_CBM)
df_CBM.head(10)


,student_id,max_question_score,question_number,question_title,question_duration_sec,auto_score_per_question,response_CBM,total_score,auto_score
11,363,2.0,2.1,H25 - CBA - Cache-miss,70,2.0,5 3,53.82,14.0
12,363,2.0,2.1,H25 - CBA - Cache-miss,70,2.0,2 2,53.82,14.0
13,363,2.0,2.1,H25 - CBA - Cache-miss,70,2.0,4 2,53.82,14.0
14,363,2.0,2.1,H25 - CBA - Cache-miss,70,2.0,3 1,53.82,14.0
15,363,2.0,2.1,H25 - CBA - Cache-miss,70,2.0,1 1,53.82,14.0
16,363,2.0,2.2,H25 - CBA - Adder,44,2.0,3 3,53.82,14.0
17,363,2.0,2.2,H25 - CBA - Adder,44,2.0,5 3,53.82,14.0
18,363,2.0,2.2,H25 - CBA - Adder,44,2.0,4 1,53.82,14.0
19,363,2.0,2.2,H25 - CBA - Adder,44,2.0,2 1,53.82,14.0
20,363,2.0,2.2,H25 - CBA - Adder,44,2.0,1 1,53.82,14.0


In [15]:
df_CBM = split_response_cbm(df_CBM, config)
df_CBM.head(10)


,student_id,max_question_score,question_number,question_title,question_duration_sec,auto_score_per_question,total_score,auto_score,answer_alternative,confidence_level
11,363,2.0,2.1,H25 - CBA - Cache-miss,70,2.0,53.82,14.0,5,3
12,363,2.0,2.1,H25 - CBA - Cache-miss,70,2.0,53.82,14.0,2,2
13,363,2.0,2.1,H25 - CBA - Cache-miss,70,2.0,53.82,14.0,4,2
14,363,2.0,2.1,H25 - CBA - Cache-miss,70,2.0,53.82,14.0,3,1
15,363,2.0,2.1,H25 - CBA - Cache-miss,70,2.0,53.82,14.0,1,1
16,363,2.0,2.2,H25 - CBA - Adder,44,2.0,53.82,14.0,3,3
17,363,2.0,2.2,H25 - CBA - Adder,44,2.0,53.82,14.0,5,3
18,363,2.0,2.2,H25 - CBA - Adder,44,2.0,53.82,14.0,4,1
19,363,2.0,2.2,H25 - CBA - Adder,44,2.0,53.82,14.0,2,1
20,363,2.0,2.2,H25 - CBA - Adder,44,2.0,53.82,14.0,1,1


In [16]:
df_CBM = map_confidence_level(df_CBM)
df_CBM.head(10)

,student_id,max_question_score,question_number,question_title,question_duration_sec,auto_score_per_question,total_score,auto_score,answer_alternative,confidence_level
11,363,2.0,2.1,H25 - CBA - Cache-miss,70,2.0,53.82,14.0,5,50.0
12,363,2.0,2.1,H25 - CBA - Cache-miss,70,2.0,53.82,14.0,2,25.0
13,363,2.0,2.1,H25 - CBA - Cache-miss,70,2.0,53.82,14.0,4,25.0
14,363,2.0,2.1,H25 - CBA - Cache-miss,70,2.0,53.82,14.0,3,0.0
15,363,2.0,2.1,H25 - CBA - Cache-miss,70,2.0,53.82,14.0,1,0.0
16,363,2.0,2.2,H25 - CBA - Adder,44,2.0,53.82,14.0,3,50.0
17,363,2.0,2.2,H25 - CBA - Adder,44,2.0,53.82,14.0,5,50.0
18,363,2.0,2.2,H25 - CBA - Adder,44,2.0,53.82,14.0,4,0.0
19,363,2.0,2.2,H25 - CBA - Adder,44,2.0,53.82,14.0,2,0.0
20,363,2.0,2.2,H25 - CBA - Adder,44,2.0,53.82,14.0,1,0.0


In [17]:
df_CBM = validate_and_remove_invalid_confidence(df_CBM)
df_CBM.head(10)

Found 60 student(s) with invalid confidence sums across 7 question(s):
  Under 100: 12 student(s)
  Over  100: 48 student(s)
  Question 2.1: 20 student(s) with incorrect sum
  Question 2.2: 23 student(s) with incorrect sum
  Question 2.3: 35 student(s) with incorrect sum
  Question 2.4: 24 student(s) with incorrect sum
  Question 2.5: 24 student(s) with incorrect sum
  Question 2.6: 31 student(s) with incorrect sum
  Question 2.7: 32 student(s) with incorrect sum


/Users/yulis/Desktop/Masteroppgave_Eksamensanalyse/utils/helper_CBM.py:183: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby([student_col, question_col])[confidence_col]
/Users/yulis/Desktop/Masteroppgave_Eksamensanalyse/utils/helper_CBM.py:197: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  invalid.groupby(question_col)[student_col]


,student_id,max_question_score,question_number,question_title,question_duration_sec,auto_score_per_question,total_score,auto_score,answer_alternative,confidence_level
0,363,2.0,2.1,H25 - CBA - Cache-miss,70,2.0,53.82,14.0,5,50.0
1,363,2.0,2.1,H25 - CBA - Cache-miss,70,2.0,53.82,14.0,2,25.0
2,363,2.0,2.1,H25 - CBA - Cache-miss,70,2.0,53.82,14.0,4,25.0
3,363,2.0,2.1,H25 - CBA - Cache-miss,70,2.0,53.82,14.0,3,0.0
4,363,2.0,2.1,H25 - CBA - Cache-miss,70,2.0,53.82,14.0,1,0.0
5,363,2.0,2.2,H25 - CBA - Adder,44,2.0,53.82,14.0,3,50.0
6,363,2.0,2.2,H25 - CBA - Adder,44,2.0,53.82,14.0,5,50.0
7,363,2.0,2.2,H25 - CBA - Adder,44,2.0,53.82,14.0,4,0.0
8,363,2.0,2.2,H25 - CBA - Adder,44,2.0,53.82,14.0,2,0.0
9,363,2.0,2.2,H25 - CBA - Adder,44,2.0,53.82,14.0,1,0.0


In [18]:
output_folder = root_path / "analysis" / "CBM" / "2025" / "Exam" / "data" / "CBM_format"
save_csv_file(df_CBM, "IN1020_2025_CBM", output_folder)

✅ DataFrame saved as: /Users/yulis/Desktop/Masteroppgave_Eksamensanalyse/analysis/CBM/2025/Exam/data/CBM_format/IN1020_2025_CBM.csv
